In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from geopy.distance import geodesic

import warnings
warnings.filterwarnings("ignore")

In [2]:
#Data Loading

In [3]:
df = pd.read_csv(r"C:\Users\SAMIKSHA\Desktop\Factory-Reallocation-Optimization\data\cleaned.csv")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead Time
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,909
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,909
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,909
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,909
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,912


In [4]:
#Factory Mapping

In [5]:
df["Order Date"] = pd.to_datetime(df["Order Date"],dayfirst = True, errors = 'coerce')
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst = True, errors='coerce')

In [6]:
factory_mapping = {

"Wonka Bar - Nutty Crunch Surprise":"Lot's O' Nuts",
"Wonka Bar - Fudge Mallows":"Lot's O' Nuts",
"Wonka Bar -Scrumdiddlyumptious":"Lot's O' Nuts",

"Wonka Bar - Milk Chocolate":"Wicked Choccy's",
"Wonka Bar - Triple Dazzle Caramel":"Wicked Choccy's",

"Laffy Taffy":"Sugar Shack",
"SweeTARTS":"Sugar Shack",
"Nerds":"Sugar Shack",
"Fun Dip":"Sugar Shack",

"Fizzy Lifting Drinks":"Sugar Shack",

"Everlasting Gobstopper":"Secret Factory",

"Hair Toffee":"The Other Factory",

"Lickable Wallpaper":"Secret Factory",
"Wonka Gum":"Secret Factory",

"Kazookles":"The Other Factory"

}

df["Factory"] = df["Product Name"].map(factory_mapping)

In [7]:
df[["Product Name","Factory"]].head()

,Product Name,Factory
0,Wonka Bar - Milk Chocolate,Wicked Choccy's
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's


In [8]:
factory_coordinates = {

"Lot's O' Nuts":(32.881893,-111.768036),

"Wicked Choccy's":(32.076176,-81.088371),

"Sugar Shack":(48.11914,-96.18115),

"Secret Factory":(41.446333,-90.565487),

"The Other Factory":(35.1175,-89.971107)

}

In [9]:
df["Factory Latitude"] = df["Factory"].apply(
    lambda x: factory_coordinates[x][0]
)

df["Factory Longitude"] = df["Factory"].apply(
    lambda x: factory_coordinates[x][1]
)

In [10]:
region_coords = {
    "Interior": (39.0997, -94.5786),   # Kansas City (central US)
    "Atlantic": (40.7128, -74.0060),   # New York City
    "Gulf": (29.7604, -95.3698),       # Houston
    "Pacific": (34.0522, -118.2437)    # Los Angeles
}


In [11]:
df["Customer Latitude"] = df["Region"].map(lambda x: region_coords[x][0])
df["Customer Longitude"] = df["Region"].map(lambda x: region_coords[x][1])

In [12]:
#Distance Calculation

In [13]:
from geopy.distance import geodesic

def calculate_distance(row):
    factory = (row["Factory Latitude"], row["Factory Longitude"])
    customer = (row["Customer Latitude"], row["Customer Longitude"])
    return geodesic(factory, customer).km

df["Shipping Distance (km)"] = df.apply(calculate_distance, axis=1)

In [14]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Units,Gross Profit,Cost,Lead Time,Factory,Factory Latitude,Factory Longitude,Customer Latitude,Customer Longitude,Shipping Distance (km)
0,1,US-2021-103800-CHO-MIL-31000,2024-03-01,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,2,4.22,2.28,909,Wicked Choccy's,32.076176,-81.088371,39.0997,-94.5786,1447.402305
1,2,US-2021-112326-CHO-TRI-54000,2024-04-01,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,2,4.90,2.60,909,Wicked Choccy's,32.076176,-81.088371,39.0997,-94.5786,1447.402305
2,3,US-2021-112326-CHO-NUT-13000,2024-04-01,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3,7.47,3.00,909,Lot's O' Nuts,32.881893,-111.768036,39.0997,-94.5786,1693.005751
3,4,US-2021-112326-CHO-SCR-58000,2024-04-01,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3,7.50,3.30,909,Lot's O' Nuts,32.881893,-111.768036,39.0997,-94.5786,1693.005751
4,5,US-2021-141817-CHO-TRI-54000,2024-05-01,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,3,7.35,3.90,912,Wicked Choccy's,32.076176,-81.088371,40.7128,-74.0060,1148.912517


In [15]:
# Calculate distance
df["Shipping Distance (km)"] = df.apply(calculate_distance, axis=1)

# Scale into a NEW column
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df["Shipping Distance Scaled"] = scaler.fit_transform(
    df[["Shipping Distance (km)"]]
)

In [16]:
df.head(1)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Gross Profit,Cost,Lead Time,Factory,Factory Latitude,Factory Longitude,Customer Latitude,Customer Longitude,Shipping Distance (km),Shipping Distance Scaled
0,1,US-2021-103800-CHO-MIL-31000,2024-03-01,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,4.22,2.28,909,Wicked Choccy's,32.076176,-81.088371,39.0997,-94.5786,1447.402305,-0.424506


In [17]:
#Label Encoding

In [18]:
from sklearn.preprocessing import LabelEncoder

ship_encoder = LabelEncoder()
factory_encoder = LabelEncoder()
product_encoder = LabelEncoder()
division_encoder = LabelEncoder()
region_encoder = LabelEncoder()

df["Ship Mode"] = ship_encoder.fit_transform(df["Ship Mode"])
df["Factory"] = factory_encoder.fit_transform(df["Factory"])
df["Product Name"] = product_encoder.fit_transform(df["Product Name"])
df["Division"] = division_encoder.fit_transform(df["Division"])
df["Region"] = region_encoder.fit_transform(df["Region"])

print("Ship Mode Mapping")
print(dict(zip(ship_encoder.classes_, ship_encoder.transform(ship_encoder.classes_))))

print("\nFactory Mapping")
print(dict(zip(factory_encoder.classes_, factory_encoder.transform(factory_encoder.classes_))))

print("\nProduct Name")
print(dict(zip(product_encoder.classes_, product_encoder.transform(product_encoder.classes_))))

print("\nDivision")
print(dict(zip(division_encoder.classes_, division_encoder.transform(division_encoder.classes_))))

print("\nRegion")
print(dict(zip(region_encoder.classes_, region_encoder.transform(region_encoder.classes_))))



Ship Mode Mapping
{'First Class': np.int64(0), 'Same Day': np.int64(1), 'Second Class': np.int64(2), 'Standard Class': np.int64(3)}

Factory Mapping
{"Lot's O' Nuts": np.int64(0), 'Secret Factory': np.int64(1), 'Sugar Shack': np.int64(2), 'The Other Factory': np.int64(3), "Wicked Choccy's": np.int64(4)}

Product Name
{'Everlasting Gobstopper': np.int64(0), 'Fizzy Lifting Drinks': np.int64(1), 'Fun Dip': np.int64(2), 'Hair Toffee': np.int64(3), 'Kazookles': np.int64(4), 'Laffy Taffy': np.int64(5), 'Lickable Wallpaper': np.int64(6), 'Nerds': np.int64(7), 'SweeTARTS': np.int64(8), 'Wonka Bar - Fudge Mallows': np.int64(9), 'Wonka Bar - Milk Chocolate': np.int64(10), 'Wonka Bar - Nutty Crunch Surprise': np.int64(11), 'Wonka Bar - Triple Dazzle Caramel': np.int64(12), 'Wonka Bar -Scrumdiddlyumptious': np.int64(13), 'Wonka Gum': np.int64(14)}

Division
{'Chocolate': np.int64(0), 'Other': np.int64(1), 'Sugar': np.int64(2)}

Region
{'Atlantic': np.int64(0), 'Gulf': np.int64(1), 'Interior': np.i

In [20]:
#Scaling
#Factor	Effect
#Base processing time	Every order takes at least 2 days
#Shipping Distance	Longer distance = more days
#Ship Mode	Faster shipping modes reduce time
#Factory Efficiency	Some factories are more efficient

In [21]:
base_time = 2

In [22]:

distance_days = df["Shipping Distance (km)"] / 600
# distance  days
#  600 km    1
#  1200 km   2
#  1800 km   3

In [23]:
ship_mode_days = df["Ship Mode"].map({
    1: 0,   # Same Day
    0: 1,   # First Class
    2: 2,   # Second Class
    3: 3    # Standard Class
})

In [24]:
factory_delay = df["Factory"].map({

    0: 0.4,   # Lot's O' Nuts
    1: 0.8,   # Secret Factory
    2: 0.2,   # Sugar Shack
    3: 0.6,   # The Other Factory
    4: 1.0    # Wicked Choccy's

})

In [25]:
region_delay = df["Region"].map({

    0: 0.5,   # Atlantic
    1: 0.8,   # Gulf
    2: 0.3,   # Interior
    3: 1.0    # Pacific

})

In [26]:
np.random.seed(42)

variation = np.random.uniform(0, 0.5, len(df))

In [27]:
#Creating bussiness rule lead time
df["Lead Time"] = (
    base_time
    + distance_days
    + ship_mode_days
    + factory_delay
    + region_delay
    + variation
).round(1)

In [28]:
print(df["Lead Time"].describe())

count    10194.000000
mean         9.072170
std          2.052552
min          4.000000
25%          7.700000
50%          8.800000
75%         10.700000
max         13.300000
Name: Lead Time, dtype: float64


In [29]:
#Applying IQR
before = len(df)

Q1 = df["Lead Time"].quantile(0.25)
Q3 = df["Lead Time"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[
    (df["Lead Time"] >= lower) &
    (df["Lead Time"] <= upper)
]

after = len(df)

print("Rows before:", before)
print("Rows after :", after)
print("Rows removed:", before - after)

Rows before: 10194
Rows after : 10194
Rows removed: 0


In [30]:
scaler = StandardScaler()

df["Sales Scaled"] = scaler.fit_transform(df[["Sales"]])
df["Units Scaled"] = scaler.fit_transform(df[["Units"]])
df["Cost Scaled"] = scaler.fit_transform(df[["Cost"]])
df["Gross Profit Scaled"] = scaler.fit_transform(df[["Gross Profit"]])


In [31]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Factory Latitude,Factory Longitude,Customer Latitude,Customer Longitude,Shipping Distance (km),Shipping Distance Scaled,Sales Scaled,Units Scaled,Cost Scaled,Gross Profit Scaled
0,1,US-2021-103800-CHO-MIL-31000,2024-03-01,2026-06-30,3,103800,United States,Houston,Texas,77095,...,32.076176,-81.088371,39.0997,-94.5786,1447.402305,-0.424506,-0.653283,-0.804161,-0.486444,-0.744565
1,2,US-2021-112326-CHO-TRI-54000,2024-04-01,2026-07-01,3,112326,United States,Naperville,Illinois,60540,...,32.076176,-81.088371,39.0997,-94.5786,1447.402305,-0.424506,-0.565104,-0.804161,-0.423220,-0.642208
2,3,US-2021-112326-CHO-NUT-13000,2024-04-01,2026-07-01,3,112326,United States,Naperville,Illinois,60540,...,32.881893,-111.768036,39.0997,-94.5786,1693.005751,-0.194790,-0.303210,-0.355370,-0.344191,-0.255358
3,4,US-2021-112326-CHO-SCR-58000,2024-04-01,2026-07-01,3,112326,United States,Naperville,Illinois,60540,...,32.881893,-111.768036,39.0997,-94.5786,1693.005751,-0.194790,-0.274110,-0.355370,-0.284919,-0.250843
4,5,US-2021-141817-CHO-TRI-54000,2024-05-01,2026-07-05,3,141817,United States,Philadelphia,Pennsylvania,19143,...,32.076176,-81.088371,40.7128,-74.0060,1148.912517,-0.703686,-0.234429,-0.355370,-0.166374,-0.273421


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Row ID                    10194 non-null  int64         
 1   Order ID                  10194 non-null  object        
 2   Order Date                4130 non-null   datetime64[ns]
 3   Ship Date                 10194 non-null  datetime64[ns]
 4   Ship Mode                 10194 non-null  int64         
 5   Customer ID               10194 non-null  int64         
 6   Country/Region            10194 non-null  object        
 7   City                      10194 non-null  object        
 8   State/Province            10194 non-null  object        
 9   Postal Code               10194 non-null  object        
 10  Division                  10194 non-null  int64         
 11  Region                    10194 non-null  int64         
 12  Product ID        

In [33]:
features = [
    "Product Name",
    "Factory",
    "Ship Mode",
    "Division",
    "Region",
    "Sales Scaled",
    "Units Scaled",
    "Cost Scaled",
    "Gross Profit Scaled",
    "Shipping Distance Scaled"
]



In [34]:
df.to_csv(
    "ml_ready_data.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


In [36]:
import joblib

joblib.dump(ship_encoder, "ship_mode_encoder.pkl")
joblib.dump(factory_encoder, "factory_encoder.pkl")
joblib.dump(region_encoder, "region_encoder.pkl")
joblib.dump(product_encoder, "product_encoder.pkl")
joblib.dump(division_encoder, "division_encoder.pkl")

['division_encoder.pkl']

In [37]:
df['Shipping Distance (km)'].unique()

array([1447.4023046 , 1693.00575057, 1148.91251749, 1596.9271132 ,
       1387.92245524, 3457.70810056, 1367.09413709, 3451.76438661,
        781.15791289,  615.67977152, 1391.81318297, 2587.91610629,
       2560.59425096,  429.33565734, 1010.35531916, 1532.0871724 ,
       2039.3039255 ,  602.34371483, 1939.18022901, 2408.70287924])

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Row ID                    10194 non-null  int64         
 1   Order ID                  10194 non-null  object        
 2   Order Date                4130 non-null   datetime64[ns]
 3   Ship Date                 10194 non-null  datetime64[ns]
 4   Ship Mode                 10194 non-null  int64         
 5   Customer ID               10194 non-null  int64         
 6   Country/Region            10194 non-null  object        
 7   City                      10194 non-null  object        
 8   State/Province            10194 non-null  object        
 9   Postal Code               10194 non-null  object        
 10  Division                  10194 non-null  int64         
 11  Region                    10194 non-null  int64         
 12  Product ID        

In [39]:
df['Ship Mode'].unique()

array([3, 0, 2, 1])

In [40]:
df['Division'].unique()

array([0, 1, 2])